## Introduction
This notebook implements a baseline and extended workflow for a news recommendation task.
The objective is to predict the next clicked article from user click history.

## Dataset Overview
The dataset includes user click logs and article metadata/embeddings.
Expected files under `../data/data_raw/`:
- train_click_log.csv
- testA_click_log.csv
- testB_click_log.csv
- articles.csv
- articles_emb.csv


In [ ]:
!pip install faiss-cpu

[This section has been converted to the English edition.]


In [ ]:
pip install protobuf==3.20.*

In [ ]:
# 
import pandas as pd  # ,.
import numpy as np  # ,.

# 
from tqdm import tqdm  # ,.
from collections import defaultdict  # ,.
import collections  # ,、.

# 
import os  # ,、.
import math  # ,、.

# 
import warnings
warnings.filterwarnings('ignore')  # ,.
import pickle  #  Python ,.

# 
import faiss  # .

# 
import random  # .
from sklearn.preprocessing import MinMaxScaler  # ,.

# 
from datetime import datetime  # ,、.

# 
import logging  # ,.
import time  # ,、.
import lightgbm as lgb  # ,.
from gensim.models import Word2Vec  #  Gensim ,.

In [ ]:

# Google Drive


In [ ]:
data_path = '../data/data_raw/'
save_path = '../data/temp_results/'

# , 
metric_recall = False

[This section has been converted to the English edition.]


In [ ]:
# debug: 
def get_all_click_sample(data_path, sample_nums=1000):
    """
        
        data_path: 
        sample_nums: （,）
    """
    all_click = pd.read_csv(data_path + 'train_click_log.csv')
    all_user_ids = all_click.user_id.unique()

    sample_user_ids = np.random.choice(all_user_ids, size=sample_nums, replace=False)
    all_click = all_click[all_click['user_id'].isin(sample_user_ids)]

    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click

# ,,
# ,
def get_all_click_df(data_path, offline=True):
    if offline:
        all_click = pd.read_csv(data_path + 'train_click_log.csv')
    else:
        trn_click = pd.read_csv(data_path + 'train_click_log.csv')
        tst_click = pd.read_csv(data_path + 'testA_click_log.csv')

        all_click = pd.concat([trn_click,tst_click])

    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click

In [ ]:
# 
def get_item_info_df(data_path):
    item_info_df = pd.read_csv(data_path + 'articles.csv')

    # click_article_id,article_idclick_article_id
    item_info_df = item_info_df.rename(columns={'article_id': 'click_article_id'})

    return item_info_df

In [ ]:
# Embedding
# （Embedding） CSV ,（embedding dictionary）
def get_item_emb_dict(data_path):
    item_emb_df = pd.read_csv(data_path + 'articles_emb.csv')

    item_emb_cols = [x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np = np.ascontiguousarray(item_emb_df[item_emb_cols])
    # 
    item_emb_np = item_emb_np / np.linalg.norm(item_emb_np, axis=1, keepdims=True)

    item_emb_dict = dict(zip(item_emb_df['article_id'], item_emb_np))
    pickle.dump(item_emb_dict, open(save_path + 'item_content_emb.pkl', 'wb'))

    return item_emb_dict


# 1. `item_emb_cols = [x for x in item_emb_df.columns if 'emb' in x]`: DataFrame  `'emb'` , `item_emb_cols` .
# 2. `item_emb_np = np.ascontiguousarray(item_emb_df[item_emb_cols])`: DataFrame  NumPy , `item_emb_np` .
# 3. `item_emb_np = item_emb_np / np.linalg.norm(item_emb_np, axis=1, keepdims=True)`:,（norm）1..
# 4. `item_emb_dict = dict(zip(item_emb_df['article_id'], item_emb_np))`:ID, `item_emb_dict` .
# 5. `pickle.dump(item_emb_dict, open(save_path + 'item_content_emb.pkl', 'wb'))`: pickle 


In [ ]:
# 
all_click_df = get_all_click_sample(data_path)


max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))

# 
# all_click_df = get_all_click_df(data_path = data_path, offline=False)

# ,
all_click_df['click_timestamp'] = all_click_df[['click_timestamp']].apply(max_min_scaler)

In [ ]:
item_info_df = get_item_info_df(data_path)

In [ ]:
item_emb_dict = get_item_emb_dict(data_path)

[This section has been converted to the English edition.]


[This section has been converted to the English edition.]


In [ ]:
#    {user1: [(item1, time1), (item2, time2)..]...}
def get_user_item_time(click_df):

    click_df = click_df.sort_values('click_timestamp')

    def make_item_time_pair(df):
        return list(zip(df['click_article_id'], df['click_timestamp']))

    user_item_time_df = click_df.groupby('user_id')[['click_article_id', 'click_timestamp']].apply(lambda x: make_item_time_pair(x))\
                                                            .reset_index().rename(columns={0: 'item_time_list'})
    user_item_time_dict = dict(zip(user_item_time_df['user_id'], user_item_time_df['item_time_list']))

    return user_item_time_dict

[This section has been converted to the English edition.]


In [ ]:
#   {item1: [(user1, time1), (user2, time2)...]...}
def get_item_user_time_dict(click_df):
    """
    ,ID,
    :param click_df: 
    :return: ID
    """
    def make_user_time_pair(df):
        return list(zip(df['user_id'], df['click_timestamp']))

    click_df = click_df.sort_values('click_timestamp')
    item_user_time_df = click_df.groupby('click_article_id')[['user_id', 'click_timestamp']].apply(lambda x: make_user_time_pair(x))\
                                                            .reset_index().rename(columns={0: 'user_time_list'})

    item_user_time_dict = dict(zip(item_user_time_df['click_article_id'], item_user_time_df['user_time_list']))
    return item_user_time_dict

[This section has been converted to the English edition.]


In [ ]:
# 
def get_hist_and_last_click(all_click):

    all_click = all_click.sort_values(by=['user_id', 'click_timestamp'])
    click_last_df = all_click.groupby('user_id').tail(1)

    # （len(user_df) == 1）,hist_func .
    #,hist_func .
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]

    click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)

    return click_hist_df, click_last_df



# 1. `click_last_df = all_click.groupby('user_id').tail(1)`:ID,.
# 2. `def hist_func(user_df):`: `hist_func`, DataFrame,.
# 3. `if len(user_df) == 1:`:1,,,.
# 4. `return user_df[:-1]`:1,,.
# 5. `click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)`:ID,
#  `hist_func` ,. `click_hist_df` .

[This section has been converted to the English edition.]


In [ ]:
# id,,,
def get_item_info_dict(item_info_df):
    max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))
    item_info_df['created_at_ts'] = item_info_df[['created_at_ts']].apply(max_min_scaler)

    item_type_dict = dict(zip(item_info_df['click_article_id'], item_info_df['category_id']))
    item_words_dict = dict(zip(item_info_df['click_article_id'], item_info_df['words_count']))
    item_created_time_dict = dict(zip(item_info_df['click_article_id'], item_info_df['created_at_ts']))

    return item_type_dict, item_words_dict, item_created_time_dict

[This section has been converted to the English edition.]


In [ ]:
def get_user_hist_item_info_dict(all_click):
    # ,:
    # 1. 
    # 2. ID
    # 3. 
    # 4. （）

    # user_id
    user_hist_item_typs = all_click.groupby('user_id')['category_id'].agg(set).reset_index()
    user_hist_item_typs_dict = dict(zip(user_hist_item_typs['user_id'], user_hist_item_typs['category_id']))
    # 1. user_id,（category_id）
    # 2. ,user_id,

    # user_id
    user_hist_item_ids_dict = all_click.groupby('user_id')['click_article_id'].agg(set).reset_index()
    user_hist_item_ids_dict = dict(zip(user_hist_item_ids_dict['user_id'], user_hist_item_ids_dict['click_article_id']))
    # 1. user_id,ID（click_article_id）
    # 2. ,user_id,ID

    # user_id
    user_hist_item_words = all_click.groupby('user_id')['words_count'].agg('mean').reset_index()
    user_hist_item_words_dict = dict(zip(user_hist_item_words['user_id'], user_hist_item_words['words_count']))
    # 1. user_id,（words_count）
    # 2. ,user_id,

    # user_id
    all_click_ = all_click.sort_values('click_timestamp')
    user_last_item_created_time = all_click_.groupby('user_id')['created_at_ts'].apply(lambda x: x.iloc[-1]).reset_index()
    # 1. （click_timestamp）
    # 2. user_id,（created_at_ts）

    max_min_scaler = lambda x : (x - np.min(x)) / (np.max(x) - np.min(x))
    user_last_item_created_time['created_at_ts'] = user_last_item_created_time[['created_at_ts']].apply(max_min_scaler)
    # 1. ,0-1
    # 2. 

    user_last_item_created_time_dict = dict(zip(user_last_item_created_time['user_id'], \
                                                user_last_item_created_time['created_at_ts']))
    # ,user_id,

    return user_hist_item_typs_dict, user_hist_item_ids_dict, user_hist_item_words_dict, user_last_item_created_time_dict
    # ,、ID、

[This section has been converted to the English edition.]


[This section has been converted to the English edition.]


In [ ]:
# 
def get_item_topk_click(click_df, k):
    topk_click = click_df['click_article_id'].value_counts().index[:k]
    return topk_click

[This section has been converted to the English edition.]


In [ ]:
# ,
item_type_dict, item_words_dict, item_created_time_dict = get_item_info_dict(item_info_df)

In [ ]:
# ,
user_multi_recall_dict =  {'itemcf_sim_itemcf_recall': {},
                           'embedding_sim_item_recall': {},
                           'cold_start_recall': {}}

[This section has been converted to the English edition.]


In [ ]:
# 
trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)

[This section has been converted to the English edition.]


In [ ]:
def metrics_recall(user_recall_items_dict, trn_last_click_df, topk=50):
    # 10, 20, 30, 40, 50（hit rate）

    # ,user_id,click_article_id
    last_click_item_dict = dict(zip(trn_last_click_df['user_id'], trn_last_click_df['click_article_id']))

    # 
    user_num = len(user_recall_items_dict)

    # 10, 20, 30, 40, 50
    for k in range(10, topk + 1, 10):
        hit_num = 0  # 

        # 
        for user, item_list in user_recall_items_dict.items():
            # k（ID）
            tmp_recall_items = [x[0] for x in user_recall_items_dict[user][:k]]

            # k,
            if last_click_item_dict[user] in set(tmp_recall_items):
                hit_num += 1

        # , =  / 
        hit_rate = round(hit_num * 1.0 / user_num, 5)

        # 
        print('topk:', k, ' | hit_num:', hit_num, ' | hit_rate:', hit_rate, ' | user_num:', user_num)

[This section has been converted to the English edition.]


[This section has been converted to the English edition.]


In [ ]:
user_item_time_dict = get_user_item_time(all_click_df)

In [ ]:
def itemcf_sim(all_click_df, option='raw'):


    # 
    # ,.
    # ij,sim(i,j)..


    user_item_time_dict = get_user_item_time(all_click_df)

    # 
    i2i_sim = defaultdict(dict)
    item_cnt = defaultdict(int)


    if option == 'raw':
        print('\n')
        # 
        # 
        for user, item_time_list in tqdm(user_item_time_dict.items(), desc="Building item-item similarity matrix (raw)"):
            # i
            for i, i_click_time in item_time_list:
                item_cnt[i] += 1  # i

                # ij
                for j, j_click_time in item_time_list:
                    if i == j:  # ij,
                        continue

                    i2i_sim[i].setdefault(j, 0)  # ij0（）

                    # ij,
                    i2i_sim[i][j] += 1 / math.log(len(item_time_list) + 1)


    elif option == 'weighted':
        print('\n')
        # 
        for user, item_time_list in tqdm(user_item_time_dict.items(), desc="Building item-item similarity matrix (weighted)"):
            for loc1, (i, i_click_time) in enumerate(item_time_list):
                item_cnt[i] += 1
                i2i_sim.setdefault(i, {})
                for loc2, (j, j_click_time) in enumerate(item_time_list):
                    if i == j:
                        continue

                    # 
                    loc_alpha = 1.0 if loc2 > loc1 else 0.7
                    # ,
                    loc_weight = loc_alpha * (0.9 ** (np.abs(loc2 - loc1) - 1))
                    # ,
                    click_time_weight = np.exp(0.7 ** np.abs(i_click_time - j_click_time))
                    # ,
                    created_time_weight = np.exp(0.8 ** np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
                    # ,
                    words_weight = np.exp(0.7 ** np.abs(item_words_dict[i] - item_words_dict[j]))
                    i2i_sim[i].setdefault(j, 0)
                    # 
                    i2i_sim[i][j] += loc_weight * click_time_weight * created_time_weight * words_weight/ math.log(len(item_time_list) + 1)

    # 
    i2i_sim_final = defaultdict(dict)
    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_final[i][j] = wij / math.sqrt(item_cnt[i] * item_cnt[j])

    # 
    with open(save_path + 'itemcf_i2i_sim.pkl', 'wb') as f:
        pickle.dump(i2i_sim_final, f)

    return i2i_sim_final

[This section has been converted to the English edition.]


[This section has been converted to the English edition.]


In [ ]:
i2i_sim = itemcf_sim(all_click_df, option='weighted')

[This section has been converted to the English edition.]


In [ ]:
def get_user_activate_degree_dict(all_click_df):
    all_click_df_ = all_click_df.groupby('user_id')['click_article_id'].count().reset_index()

    # 
    mm = MinMaxScaler()
    all_click_df_['click_article_id'] = mm.fit_transform(all_click_df_[['click_article_id']])
    user_activate_degree_dict = dict(zip(all_click_df_['user_id'], all_click_df_['click_article_id']))
    # ,, [0, 1] ,.
    return user_activate_degree_dict

In [ ]:
def usercf_sim(all_click_df, user_activate_degree_dict):
    """
        
        :param all_click_df: 
        :param user_activate_degree_dict: 
        return 
    """

    # ,ID,
    item_user_time_dict = get_item_user_time_dict(all_click_df)

    # 
    u2u_sim = {}
    # 
    user_cnt = defaultdict(int)

    # 
    for item, user_time_list in tqdm(item_user_time_dict.items()):
        # 
        for u, click_time in user_time_list:
            # 
            user_cnt[u] += 1
            # ,
            u2u_sim.setdefault(u, {})
            # （,）
            for v, click_time in user_time_list:
                # ,0
                u2u_sim[u].setdefault(v, 0)
                # ,（）
                if u == v:
                    continue
                # ,
                activate_weight = 100 * 0.5 * (user_activate_degree_dict[u] + user_activate_degree_dict[v])
                # user_activate_degree_dict[u]  user_activate_degree_dict[v]  u  v （）.
                # 
                u2u_sim[u][v] += activate_weight / math.log(len(user_time_list) + 1)

    # ,
    u2u_sim_ = u2u_sim.copy()
    # 
    for u, related_users in u2u_sim.items():
        for v, wij in related_users.items():
            u2u_sim_[u][v] = wij / math.sqrt(user_cnt[u] * user_cnt[v])

    # 
    pickle.dump(u2u_sim_, open(save_path + 'usercf_u2u_sim.pkl', 'wb'))

    return u2u_sim_


In [ ]:
# usercf,
# ,
user_activate_degree_dict = get_user_activate_degree_dict(all_click_df)
u2u_sim = usercf_sim(all_click_df, user_activate_degree_dict)

[This section has been converted to the English edition.]


[This section has been converted to the English edition.]


In [ ]:
import numpy as np
import faiss
import collections
import pickle
from tqdm import tqdm

def embdding_sim(click_df, item_emb_df, save_path, topk):
    """
        embedding
        :param click_df: 
        :param item_emb_df: embedding
        :param save_path: 
        :patam topk: topk
        return 

        : , embeddingtopk, ,faiss
    """

    # id
    item_idx_2_rawid_dict = dict(zip(item_emb_df.index, item_emb_df['article_id']))

    item_emb_cols = [x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np = np.ascontiguousarray(item_emb_df[item_emb_cols].values, dtype=np.float32)
    # 
    item_emb_np = item_emb_np / np.linalg.norm(item_emb_np, axis=1, keepdims=True)

    # faiss
    item_index = faiss.IndexFlatIP(item_emb_np.shape[1])   #  faiss ,IndexFlatIP （Inner Product）, embedding .
    item_index.add(item_emb_np)
    sim, idx = item_index.search(item_emb_np, topk)  # 

    # id
    item_sim_dict = collections.defaultdict(dict)

    # tqdm
    for target_idx, sim_value_list, rele_idx_list in tqdm(zip(range(len(item_emb_np)), sim, idx), total=len(item_emb_np)):
        target_raw_id = item_idx_2_rawid_dict[target_idx]
        # 1, topk-1
        for rele_idx, sim_value in zip(rele_idx_list[1:], sim_value_list[1:]):
            rele_raw_id = item_idx_2_rawid_dict[rele_idx]
            item_sim_dict[target_raw_id][rele_raw_id] = item_sim_dict.get(target_raw_id, {}).get(rele_raw_id, 0) + sim_value

    # i2i
    pickle.dump(item_sim_dict, open(save_path + 'emb_i2i_sim.pkl', 'wb'))  #  pickle .

    return item_sim_dict

In [ ]:
item_emb_df = pd.read_csv(data_path + '/articles_emb.csv')
# emb_i2i_sim = embdding_sim(all_click_df, item_emb_df, save_path, topk=10) # topk

[This section has been converted to the English edition.]


In [ ]:
# i2i
def content_based_recommend(user_id, user_item_time_dict, i2i_sim, sim_item_topk, recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim):
    """
        
        :param user_id: id
        :param user_item_time_dict: ,    {user1: [(item1, time1), (item2, time2)..]...}
        :param i2i_sim: ,
        :param sim_item_topk: , k
        :param recall_item_num: , 
        :param item_topk_click: ,,
        :param emb_i2i_sim: embedding

        return:  [(item1, score1), (item2, score2)...]
    """
    # 
    user_hist_items = user_item_time_dict[user_id]
    user_hist_items_ = {user_id for user_id, _ in user_hist_items}

    item_rank = {}
    for loc, (i, click_time) in enumerate(user_hist_items):
        for j, wij in sorted(i2i_sim[i].items(), key=lambda x: x[1], reverse=True)[:sim_item_topk]:
            if j in user_hist_items_:
                continue

            # 
            created_time_weight = np.exp(0.8 ** np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
            # 
            loc_weight = (0.9 ** (len(user_hist_items) - loc))

            content_weight = 1.0
            if emb_i2i_sim.get(i, {}).get(j, None) is not None:
                content_weight += emb_i2i_sim[i][j]
            if emb_i2i_sim.get(j, {}).get(i, None) is not None:
                content_weight += emb_i2i_sim[j][i]

            item_rank.setdefault(j, 0)
            item_rank[j] += created_time_weight * loc_weight * content_weight * wij

    # 10,
    if len(item_rank) < recall_item_num:
        for i, item in enumerate(item_topk_click):
            if item in item_rank.items(): # item
                continue
            item_rank[item] = - i - 100 # 
            if len(item_rank) == recall_item_num:
                break

    item_rank = sorted(item_rank.items(), key=lambda x: x[1], reverse=True)[:recall_item_num]

    return item_rank

[This section has been converted to the English edition.]


[This section has been converted to the English edition.]


In [ ]:
# itemcf, ,

if metric_recall:
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df = all_click_df

user_recall_items_dict = collections.defaultdict(dict)
user_item_time_dict = get_user_item_time(trn_hist_click_df)

i2i_sim = pickle.load(open(save_path + 'itemcf_i2i_sim.pkl', 'rb'))
emb_i2i_sim = pickle.load(open(save_path + 'emb_i2i_sim.pkl', 'rb'))

sim_item_topk = 20
recall_item_num = 10
item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)

np.random.seed(42)  # 
for user in tqdm(trn_hist_click_df['user_id'].sample(500).unique()):
    user_recall_items_dict[user] = content_based_recommend(user, user_item_time_dict, \
                                                        i2i_sim, sim_item_topk, recall_item_num, \
                                                        item_topk_click, item_created_time_dict, emb_i2i_sim)

user_multi_recall_dict['itemcf_sim_itemcf_recall'] = user_recall_items_dict
pickle.dump(user_multi_recall_dict['itemcf_sim_itemcf_recall'], open(save_path + 'itemcf_recall_dict.pkl', 'wb'))

if metric_recall:
    # 
    metrics_recall(user_multi_recall_dict['itemcf_sim_itemcf_recall'], trn_last_click_df, topk=recall_item_num)

[This section has been converted to the English edition.]


In [ ]:
# ,
if metric_recall:
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df = all_click_df

user_recall_items_dict = collections.defaultdict(dict)
user_item_time_dict = get_user_item_time(trn_hist_click_df)
i2i_sim = pickle.load(open(save_path + 'emb_i2i_sim.pkl','rb'))

sim_item_topk = 20
recall_item_num = 10

item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)

np.random.seed(42)  # 
for user in tqdm(trn_hist_click_df['user_id'].sample(500).unique()):
    user_recall_items_dict[user] = content_based_recommend(user, user_item_time_dict, i2i_sim, sim_item_topk,
                                                        recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)

user_multi_recall_dict['embedding_sim_item_recall'] = user_recall_items_dict
pickle.dump(user_multi_recall_dict['embedding_sim_item_recall'], open(save_path + 'embedding_sim_item_recall.pkl', 'wb'))

if metric_recall:
    # 
    metrics_recall(user_multi_recall_dict['embedding_sim_item_recall'], trn_last_click_df, topk=recall_item_num)

[This section has been converted to the English edition.]


In [ ]:
#  u2u2i
def user_based_recommend(user_id, user_item_time_dict, u2u_sim, sim_user_topk, recall_item_num,
                         item_topk_click, item_created_time_dict, emb_i2i_sim):
    """
        
        :param user_id: id
        :param user_item_time_dict: ,    {user1: [(item1, time1), (item2, time2)..]...}
        :param u2u_sim: ,
        :param sim_user_topk: , k
        :param recall_item_num: , 
        :param item_topk_click: ,,
        :param item_created_time_dict: 
        :param emb_i2i_sim: embedding

        return:  [(item1, score1), (item2, score2)...]
    """
    # 
    user_item_time_list = user_item_time_dict[user_id]    #  [(item1, time1), (item2, time2)..]
    user_hist_items = set([i for i, t in user_item_time_list])   # , 

    items_rank = {}
    for sim_u, wuv in sorted(u2u_sim[user_id].items(), key=lambda x: x[1], reverse=True)[:sim_user_topk]:
        for i, click_time in user_item_time_dict[sim_u]:
            if i in user_hist_items:
                continue
            items_rank.setdefault(i, 0)

            loc_weight = 1.0
            content_weight = 1.0
            created_time_weight = 1.0

            # 
            for loc, (j, click_time) in enumerate(user_item_time_list):
                # 
                loc_weight += 0.9 ** (len(user_item_time_list) - loc)
                # 
                if emb_i2i_sim.get(i, {}).get(j, None) is not None:
                    content_weight += emb_i2i_sim[i][j]
                if emb_i2i_sim.get(j, {}).get(i, None) is not None:
                    content_weight += emb_i2i_sim[j][i]

                # 
                created_time_weight += np.exp(0.8 * np.abs(item_created_time_dict[i] - item_created_time_dict[j]))

            items_rank[i] += loc_weight * content_weight * created_time_weight * wuv

    # 
    if len(items_rank) < recall_item_num:
        for i, item in enumerate(item_topk_click):
            if item in items_rank.items(): # item
                continue
            items_rank[item] = - i - 100 # 
            if len(items_rank) == recall_item_num:
                break

    items_rank = sorted(items_rank.items(), key=lambda x: x[1], reverse=True)[:recall_item_num]

    return items_rank

[This section has been converted to the English edition.]


In [ ]:
# ,
# usercfuser,,
if metric_recall:
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df = all_click_df

user_recall_items_dict = collections.defaultdict(dict)
user_item_time_dict = get_user_item_time(trn_hist_click_df)

u2u_sim = pickle.load(open(save_path + 'usercf_u2u_sim.pkl', 'rb'))

sim_user_topk = 20
recall_item_num = 10
item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)


np.random.seed(42)  # 
for user in tqdm(trn_hist_click_df['user_id'].sample(500).unique()):
    user_recall_items_dict[user] = user_based_recommend(user, user_item_time_dict, u2u_sim, sim_user_topk, \
                                                        recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)

pickle.dump(user_recall_items_dict, open(save_path + 'usercf_u2u2i_recall.pkl', 'wb'))

if metric_recall:
    # 
    metrics_recall(user_recall_items_dict, trn_last_click_df, topk=recall_item_num)

[This section has been converted to the English edition.]


[This section has been converted to the English edition.]


In [ ]:
# itemcf,,
trn_hist_click_df = all_click_df

user_recall_items_dict = collections.defaultdict(dict)
user_item_time_dict = get_user_item_time(trn_hist_click_df)
i2i_sim = pickle.load(open(save_path + 'emb_i2i_sim.pkl','rb'))

sim_item_topk = 150
recall_item_num = 100 # ,

np.random.seed(42)  # 
item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)
for user in tqdm(trn_hist_click_df['user_id'].sample(500).unique()):
    user_recall_items_dict[user] = content_based_recommend(user, user_item_time_dict, i2i_sim, sim_item_topk,
                                                        recall_item_num, item_topk_click,item_created_time_dict, emb_i2i_sim)
pickle.dump(user_recall_items_dict, open(save_path + 'cold_start_items_raw_dict.pkl', 'wb'))

In [ ]:
# 
# 
# 
# 
# 

def get_click_article_ids_set(all_click_df):
    return set(all_click_df.click_article_id.values)

def cold_start_items(user_recall_items_dict, user_hist_item_typs_dict, user_hist_item_words_dict, \
                     user_last_item_created_time_dict, item_type_dict, item_words_dict,
                     item_created_time_dict, click_article_ids_set, recall_item_num):
    """
        
        :param user_recall_items_dict: embedding, , {user1: [(item1, item2), ..], }
        :param user_hist_item_typs_dict: , 
        :param user_hist_item_words_dict: , 
        :param user_last_item_created_time_idct: ,
        :param item_tpye_idct: ,
        :param item_words_dict: ,
        :param item_created_time_dict: , 
        :param click_article_ids_set: ,, 
        :param recall_item_num: , 
    """

    cold_start_user_items_dict = {}
    for user, item_list in tqdm(user_recall_items_dict.items()):
        cold_start_user_items_dict.setdefault(user, [])
        for item, score in item_list:
            # 
            hist_item_type_set = user_hist_item_typs_dict[user]
            hist_mean_words = user_hist_item_words_dict[user]
            hist_last_item_created_time = user_last_item_created_time_dict[user]
            hist_last_item_created_time = datetime.fromtimestamp(hist_last_item_created_time)

            # 
            curr_item_type = item_type_dict[item]
            curr_item_words = item_words_dict[item]
            curr_item_created_time = item_created_time_dict[item]
            curr_item_created_time = datetime.fromtimestamp(curr_item_created_time)

            # ,, ,,
            if curr_item_type not in hist_item_type_set or \
                item in click_article_ids_set or \
                abs(curr_item_words - hist_mean_words) > 200 or \
                abs((curr_item_created_time - hist_last_item_created_time).days) > 90:
                continue

            cold_start_user_items_dict[user].append((item, score))      # {user1: [(item1, score1), (item2, score2)..]...}

    # 
    cold_start_user_items_dict = {k: sorted(v, key=lambda x:x[1], reverse=True)[:recall_item_num] \
                                  for k, v in cold_start_user_items_dict.items()}

    pickle.dump(cold_start_user_items_dict, open(save_path + 'cold_start_user_items_dict.pkl', 'wb'))

    return cold_start_user_items_dict

[This section has been converted to the English edition.]


In [ ]:
all_click_df_ = all_click_df.copy()
all_click_df_ = all_click_df_.merge(item_info_df, how='left', on='click_article_id')
user_hist_item_typs_dict, user_hist_item_ids_dict, user_hist_item_words_dict, user_last_item_created_time_dict = get_user_hist_item_info_dict(all_click_df_)
click_article_ids_set = get_click_article_ids_set(all_click_df)
# 
# ,,
cold_start_user_items_dict = cold_start_items(user_recall_items_dict, user_hist_item_typs_dict, user_hist_item_words_dict, \
                                              user_last_item_created_time_dict, item_type_dict, item_words_dict, \
                                              item_created_time_dict, click_article_ids_set, recall_item_num)

user_multi_recall_dict['cold_start_recall'] = cold_start_user_items_dict

[This section has been converted to the English edition.]


In [ ]:
def combine_recall_results(user_multi_recall_dict, weight_dict=None, topk=25):
    final_recall_items_dict = {}

    # ,,
    def norm_user_recall_items_sim(sorted_item_list):
        # ,,,
        # , 
        if len(sorted_item_list) < 2:
            return sorted_item_list

        min_sim = sorted_item_list[-1][1]
        max_sim = sorted_item_list[0][1]

        norm_sorted_item_list = []
        for item, score in sorted_item_list:
            if max_sim > 0:
                norm_score = 1.0 * (score - min_sim) / (max_sim - min_sim) if max_sim > min_sim else 1.0
            else:
                norm_score = 0.0
            norm_sorted_item_list.append((item, norm_score))

        return norm_sorted_item_list

    print('...')
    for method, user_recall_items in tqdm(user_multi_recall_dict.items()):
        print(method + '...')
        # ,
        if weight_dict == None:
            recall_method_weight = 1
        else:
            recall_method_weight = weight_dict[method]

        for user_id, sorted_item_list in user_recall_items.items(): # 
            user_recall_items[user_id] = norm_user_recall_items_sim(sorted_item_list)

        for user_id, sorted_item_list in user_recall_items.items():
            # print('user_id')
            final_recall_items_dict.setdefault(user_id, {})
            for item, score in sorted_item_list:
                final_recall_items_dict[user_id].setdefault(item, 0)
                final_recall_items_dict[user_id][item] += recall_method_weight * score

    final_recall_items_dict_rank = {}
    # 
    for user, recall_item_dict in final_recall_items_dict.items():
        final_recall_items_dict_rank[user] = sorted(recall_item_dict.items(), key=lambda x: x[1], reverse=True)[:topk]

    # 
    pickle.dump(final_recall_items_dict_rank, open(os.path.join(save_path, 'final_recall_items_dict.pkl'),'wb'))

    return final_recall_items_dict_rank

In [ ]:
# ,
weight_dict = {'itemcf_sim_itemcf_recall': 1.0,
               'embedding_sim_item_recall': 1.0,
               'cold_start_recall': 1.0}

In [ ]:
# 5
final_recall_items_dict_rank = combine_recall_results(user_multi_recall_dict, weight_dict, topk=150)

In [ ]:
multi_recall_score_list = []

for user, items in tqdm(final_recall_items_dict_rank.items()):
    if isinstance(items, (list, tuple)):  # items
        for item, score in items:
            multi_recall_score_list.append([user, item, score])
    else:
        print(f"Error: items for user {user} is not iterable")

recall_df = pd.DataFrame(multi_recall_score_list, columns=['user_id', 'click_article_id', 'pred_score'])


In [ ]:
def submit(recall_df, topk=5, model_name=None):
    # 'user_id''pred_score'DataFrame
    recall_df = recall_df.sort_values(by=['user_id', 'pred_score'])

    # 'rank',
    recall_df['rank'] = recall_df.groupby(['user_id'])['pred_score'].rank(ascending=False, method='first')

    # 'topk'
    tmp = recall_df.groupby('user_id').apply(lambda x: x['rank'].max())
    assert tmp.min() >= topk

    # 'pred_score'
    del recall_df['pred_score']

    # 'topk',DataFrame
    submit = recall_df[recall_df['rank'] <= topk].set_index(['user_id', 'rank']).unstack(-1).reset_index()

    # 
    submit.columns = [int(col) if isinstance(col, int) else col for col in submit.columns.droplevel(0)]
    submit = submit.rename(columns={'': 'user_id', 1: 'article_1', 2: 'article_2',
                                    3: 'article_3', 4: 'article_4', 5: 'article_5'})

    # CSV
    save_name = save_path + model_name + '_' + datetime.today().strftime('%m-%d') + '.csv'

    # DataFrameCSV,,
    submit.to_csv(save_name, index=False, header=True)

    return submit


In [ ]:
recall_df['click_article_id'] = recall_df['click_article_id'].astype(int)
# 
submit(recall_df, topk=5, model_name='multiple_recall')